In [30]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("ahmedshahriarsakib/usa-real-estate-dataset")

print("Path to dataset files:", path)
# The dataset is now downloaded and ready to use.

Using Colab cache for faster access to the 'usa-real-estate-dataset' dataset.
Path to dataset files: /kaggle/input/usa-real-estate-dataset


In [31]:
# Install KaggleHub if needed
!pip -q install kagglehub

import os
import glob
import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Data source:
# https://www.kaggle.com/datasets/ahmedshahriarsakib/usa-real-estate-dataset

# Download the dataset
dataset_path = kagglehub.dataset_download(
    "ahmedshahriarsakib/usa-real-estate-dataset"
)

print("Dataset downloaded to:", dataset_path)

# Locate the CSV file
if os.path.isfile(dataset_path):
    csv_file = dataset_path
else:
    csv_file = os.path.join(dataset_path, "realtor-data.zip.csv")

    if not os.path.exists(csv_file):
        csv_files = glob.glob(
            os.path.join(dataset_path, "**", "*.csv"),
            recursive=True
        )

        if len(csv_files) == 0:
            raise FileNotFoundError("No CSV file was found.")

        csv_file = csv_files[0]

print("Using file:", csv_file)

# Load only the columns needed for this assignment
df = pd.read_csv(
    csv_file,
    usecols=["price", "house_size", "state"],
    low_memory=False
)

print("Original number of records:", len(df))

# Use a random sample to make the model run efficiently in Colab
# The sample is still much larger than the required 100+ records.
MAX_ROWS = 100_000

if len(df) > MAX_ROWS:
    df = df.sample(n=MAX_ROWS, random_state=42)

# Rename columns
df = df.rename(columns={
    "house_size": "square_footage",
    "state": "location"
})

# Convert numerical columns to numeric values
df["price"] = pd.to_numeric(df["price"], errors="coerce")
df["square_footage"] = pd.to_numeric(
    df["square_footage"],
    errors="coerce"
)

# Clean the location column
df["location"] = df["location"].astype("string").str.strip()

# Remove missing, zero, and negative values
df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna(subset=["price", "square_footage", "location"])

df = df[
    (df["price"] > 0) &
    (df["square_footage"] > 0) &
    (df["location"] != "")
]

# Remove extreme outliers using the IQR method
def remove_iqr_outliers(data, column):
    q1 = data[column].quantile(0.25)
    q3 = data[column].quantile(0.75)
    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    return data[
        (data[column] >= lower_bound) &
        (data[column] <= upper_bound)
    ]

df = remove_iqr_outliers(df, "square_footage")
df = remove_iqr_outliers(df, "price")

if len(df) < 100:
    raise ValueError("Fewer than 100 records remain after cleaning.")

print("Records used for modeling:", len(df))
print("Number of locations:", df["location"].nunique())

display(df.head())
display(df.describe())

Using Colab cache for faster access to the 'usa-real-estate-dataset' dataset.
Dataset downloaded to: /kaggle/input/usa-real-estate-dataset
Using file: /kaggle/input/usa-real-estate-dataset/realtor-data.zip.csv
Original number of records: 2226382
Records used for modeling: 66072
Number of locations: 54


,price,location,square_footage
1696936,275000.0,Florida,846.0
2092671,399900.0,California,667.0
1424136,325000.0,Massachusetts,1409.0
2159466,265000.0,California,901.0
948803,292000.0,Nebraska,1949.0


,price,square_footage
count,6.607200e+04,66072.000000
mean,3.827323e+05,1795.867342
std,2.187455e+05,722.958860
min,1.000000e+00,100.000000
25%,2.200000e+05,1255.000000
50%,3.449000e+05,1674.000000
75%,5.000000e+05,2224.000000
max,1.055000e+06,4075.000000


In [32]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Features and target
X = df[["square_footage", "location"]]
y = df["price"]

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

# Make the encoder compatible with different scikit-learn versions
try:
    encoder = OneHotEncoder(
        handle_unknown="ignore",
        drop="first",
        sparse_output=False
    )
except TypeError:
    encoder = OneHotEncoder(
        handle_unknown="ignore",
        drop="first",
        sparse=False
    )

# One-hot encode location and keep square footage numeric
preprocessor = ColumnTransformer(
    transformers=[
        ("location", encoder, ["location"])
    ],
    remainder="passthrough"
)

# Create the model pipeline
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression())
])

# Train the model
model.fit(X_train, y_train)

# Evaluate the model
test_predictions = model.predict(X_test)

mae = mean_absolute_error(y_test, test_predictions)
rmse = np.sqrt(mean_squared_error(y_test, test_predictions))
r2 = r2_score(y_test, test_predictions)

print("Model Evaluation")
print("----------------")
print(f"Mean Absolute Error: ${mae:,.2f}")
print(f"Root Mean Squared Error: ${rmse:,.2f}")
print(f"R-squared Score: {r2:.3f}")

Model Evaluation
----------------
Mean Absolute Error: $118,452.88
Root Mean Squared Error: $161,157.52
R-squared Score: 0.459


In [33]:

requested_location = "California"

available_locations = set(df["location"].unique())


if requested_location not in available_locations:
    requested_location = df["location"].mode().iloc[0]
    print(
        "California was not found. Using the most common location:",
        requested_location
    )

# Create a new house with 2,000 square feet
new_house = pd.DataFrame({
    "square_footage": [2000],
    "location": [requested_location]
})

# Make the prediction
predicted_price = model.predict(new_house)[0]

print(
    f"Predicted price for a 2,000 square-foot house "
    f"in {requested_location}: ${predicted_price:,.2f}"
)

Predicted price for a 2,000 square-foot house in California: $664,532.86


In [34]:
# Get the names of all transformed features
feature_names = (
    model.named_steps["preprocessor"]
    .get_feature_names_out()
)

coefficients = model.named_steps["regressor"].coef_

coefficient_table = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coefficients
})

# Make the feature names easier to read
coefficient_table["feature"] = (
    coefficient_table["feature"]
    .str.replace("location__location_", "location=", regex=False)
    .str.replace("remainder__", "", regex=False)
)

coefficient_table["absolute_coefficient"] = (
    coefficient_table["coefficient"].abs()
)

coefficient_table = coefficient_table.sort_values(
    by="absolute_coefficient",
    ascending=False
)

print("Model Coefficients")
print("------------------")

print(
    coefficient_table[["feature", "coefficient"]].to_string(
        index=False,
        formatters={
            "coefficient": lambda value: f"{value:,.2f}"
        }
    )
)

# Display the square-footage coefficient separately
square_footage_coefficient = coefficient_table[
    coefficient_table["feature"] == "square_footage"
]["coefficient"]

if len(square_footage_coefficient) > 0:
    print(
        f"\nEstimated price change per additional square foot: "
        f"${square_footage_coefficient.iloc[0]:,.2f}"
    )

# Explain the reference location
location_encoder = (
    model.named_steps["preprocessor"]
    .named_transformers_["location"]
)

if location_encoder.drop_idx_ is not None:
    reference_index = location_encoder.drop_idx_[0]
    reference_location = location_encoder.categories_[0][reference_index]

    print(
        f"\nLocation coefficients are compared with the reference "
        f"location: {reference_location}"
    )

Model Coefficients
------------------
                      feature coefficient
              location=Hawaii  450,863.47
location=District of Columbia  387,781.99
          location=California  376,352.20
          location=Washington  280,584.63
       location=Massachusetts  266,507.90
                location=Guam  258,581.73
              location=Oregon  246,134.42
            location=Colorado  238,141.94
              location=Nevada  227,783.13
                location=Utah  211,005.34
               location=Idaho  210,083.72
             location=Montana  174,216.98
             location=Arizona  173,436.32
        location=Rhode Island  162,656.87
               location=Maine  155,658.08
              location=Alaska  153,581.06
             location=Florida  152,438.16
       location=New Hampshire  147,407.49
          location=New Jersey  140,953.40
            location=Virginia  131,532.73
         location=Connecticut  121,759.55
            location=New York  120,566